# Ähnlichkeitssuche in Python

Dasselbe wie die Live-Demo in Java, diesmal in Python: dasselbe Embedding-Modell `multilingual-e5-small`, dieselben 643 Chunks aus AstroPlanners Wissensbasis, die Suche als Schleife über alle Chunks. Die Vektoren der Chunks sind vorberechnet und liegen im Ordner `index/`. Hier wird nur der Suchtext eingebettet.

### Aufgaben

**Für alle** (rund 12 Minuten, dazu die Ladezeit)

1. Ein Text wird ein Vektor: ausführen und ablesen *(Einstieg)*
2. Zwei Texte vergleichen: erst vermuten, welches Paar am nächsten liegt, dann eigene Paare
3. Die Suchen aus der Live-Demo und eine eigene Suche

**Bonus**

4. Suchen ohne die Präfixe `query: ` und `passage: `
5. Knobelaufgabe: Die Suche soll zur Frage nach dem Alter des Universums nichts liefern.

Zeitbox 15 Minuten, zu zweit. Am schnellsten mit *Laufzeit → Alle ausführen*, dann von oben lesen. Jede Aufgabe steht noch einmal an der Stelle, an der ihr sie macht.

## Für alle

> **Aufgabe 1** (Einstieg): *Laufzeit → Alle ausführen*, dann von oben lesen. Bevor ihr die Ausgabe von Schritt 2 anseht, vermuten: Bekommt ein einzelnes Wort weniger Zahlen als eine ganze Frage?

### 1. Modell und Chunks laden

Beim ersten Mal lädt das Modell rund 490 MB von Hugging Face. Die Chunks und ihre Vektoren kommen aus dem Ordner.

In [ ]:
# Nur in Colab: Hilfsdateien und Daten aus dem Repo holen. Lokal tut die Zelle nichts.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repo = "/content/ki-workshop-handson"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dprinz/ki-workshop-handson.git", repo], check=True)
    os.chdir(f"{repo}/07-aehnlichkeitssuche")

In [1]:
import time

import numpy as np

from helfer import chunk_text, einbetten, lade_chunks, lade_index, lade_modell, zeige

start = time.perf_counter()
lade_modell()
chunks = lade_chunks()
vektoren = lade_index()  # eine Zeile pro Chunk, je 384 Zahlen
print(f"{len(chunks)} Chunks, Index {vektoren.shape}, geladen in {time.perf_counter() - start:.1f} s")

643 Chunks, Index (643, 384), geladen in 4.0 s


Ein Chunk ist ein Abschnitt eines Dokuments mit höchstens 250 Wörtern. Die erste Zeile nennt Dokument und Abschnitt. So hat das Modell den Chunk bekommen, mit dem Präfix `passage: ` davor:

In [2]:
chunk = next(c for c in chunks if c["abschnitt"] == "Astronomical twilight")
print("passage: " + chunk_text(chunk))

passage: Twilight – Astronomical twilight
Astronomical twilight occurs when the geometric center of the Sun is between 18° and 12° below the horizon. During astronomical twilight, the sky is dark enough to permit astronomical observation of point sources of light such as stars, except in regions with more intense skyglow due to light pollution, moonlight, auroras, and other sources of light. Some critical observations, such as of faint diffuse items such as nebulae and galaxies, may require observation beyond the limit of astronomical twilight. Theoretically, the faintest stars detectable by the naked eye (those of approximately the sixth magnitude) will become visible in the evening at astronomical dusk, and become invisible at astronomical dawn.


### 2. Ein Text wird ein Vektor

Eine Frage bekommt das Präfix `query: `. So wurde e5 trainiert: Fragen mit `query: `, Texte mit `passage: `.

`einbetten` steht in `helfer.py`: Tokenizer, Modell, dann der Mittelwert über die Vektoren aller Token (Mean Pooling), auf Länge 1 normiert. Wie in Java und AstroPlanner.

In [3]:
for text in ["query: Wann beginnt die astronomische Dämmerung?", "query: Dämmerung"]:
    vektor = einbetten(text)
    print(f"{text}\nDimension {len(vektor)}, Länge {np.linalg.norm(vektor):.3f}")
    print(np.round(vektor[:8], 4), "…\n")

query: Wann beginnt die astronomische Dämmerung?
Dimension 384, Länge 1.000
[ 0.1014 -0.0272 -0.0424 -0.0483  0.0687 -0.06   -0.0084  0.0564] …

query: Dämmerung
Dimension 384, Länge 1.000
[ 0.0505 -0.0321 -0.0474 -0.1057  0.0572 -0.0415  0.0274  0.04  ] …



Eine ganze Frage und ein einzelnes Wort ergeben gleich viele Zahlen: 384. Die einzelnen Zahlen haben keine lesbare Bedeutung.

Java zeigte für die erste Frage `[0.0949, -0.0175, -0.0422, -0.0464, 0.0714, …]`. Die Zahlen hier weichen etwas ab: Java lädt eine quantisierte Fassung des Modells (ONNX, 118 MB), hier läuft das Original (470 MB).

### 3. Zwei Texte vergleichen

Wie nah sich zwei Texte sind, misst die Suche am Winkel zwischen ihren Vektoren. Die Vektoren haben Länge 1, das Skalarprodukt ist deshalb schon der Kosinus des Winkels. Die Distanz ist 1 − Kosinus, wie in Java und pgvector: 0 heisst gleiche Richtung, kleiner ist näher.

Zum Vergleichen zweier Texte bekommen beide das Präfix `query: `, so empfiehlt es die Modellkarte von e5 für Ähnlichkeit zwischen gleichartigen Texten.

> **Aufgabe 2**: Bevor ihr die Ausgabe anseht, vermuten: Welches Paar unten liegt am nächsten, welches am weitesten weg? Und wo landet „twilight“, das englische Wort für Dämmerung? Dann nach der Auswertung eigene Paare ausprobieren.

In [4]:
def abstand(text_a, text_b):
    a = einbetten("query: " + text_a)
    b = einbetten("query: " + text_b)
    return 1 - np.dot(a, b)  # beide Vektoren haben Länge 1: Skalarprodukt = Kosinus


paare = [
    ("Dämmerung", "Abenddämmerung"),
    ("Dämmerung", "Sonnenuntergang"),
    ("Dämmerung", "twilight"),
    ("Dämmerung", "Pizza"),
    ("Dämmerung", "NullPointerException"),
    ("Wann beginnt die astronomische Dämmerung?", "Wann wird es nachts dunkel genug für Sterne?"),
    ("Wann beginnt die astronomische Dämmerung?", "Wie behebe ich eine NullPointerException?"),
]

ergebnisse = sorted((abstand(a, b), a, b) for a, b in paare)  # sortiert nach Distanz, das nächste Paar zuerst
for distanz, a, b in ergebnisse:
    print(f"{distanz:.3f}  {a}  |  {b}")

0.039  Dämmerung  |  Abenddämmerung
0.103  Wann beginnt die astronomische Dämmerung?  |  Wann wird es nachts dunkel genug für Sterne?
0.117  Dämmerung  |  Sonnenuntergang
0.164  Dämmerung  |  Pizza
0.165  Dämmerung  |  twilight
0.202  Dämmerung  |  NullPointerException
0.261  Wann beginnt die astronomische Dämmerung?  |  Wie behebe ich eine NullPointerException?


Was die Zahlen zeigen:

- Am nächsten liegt das Paar, das ein Wort teilt: „Dämmerung“ und „Abenddämmerung“, 0,039.
- Die zwei Fragen zur Dunkelheit haben ausser „Wann“ kein Wort gemeinsam und liegen trotzdem bei 0,103, näher als „Dämmerung“ und „Sonnenuntergang“ (0,117). Die Frage zur NullPointerException liegt mit 0,261 am weitesten weg.
- Bei einzelnen Wörtern ist die Reihenfolge weniger verlässlich: „twilight“ liegt mit 0,165 gleich weit weg wie „Pizza“ (0,164), obwohl e5 mehrsprachig ist und deutsche Fragen englische Chunks finden.
- Alle Werte liegen zwischen 0 und 0,3, auch Pizza. Bei e5 ist das so gewollt: Laut Modellkarte liegt der Kosinus meist zwischen 0,7 und 1, und es zählt die Reihenfolge, nicht der Wert. Eine Distanz von 0,2 heisst also nicht „80 % ähnlich“, sie sagt nur im Vergleich mit anderen Distanzen desselben Modells etwas.

Eigene Paare, Ideen: „Teleskop“ mit „Fernrohr“ und mit „Mikroskop“, „Bank“ mit „Sitzbank“ und mit „Geldinstitut“, ein Satz aus eurem Alltag und derselbe Satz auf Englisch.

In [5]:
meine_paare = [
    ("Teleskop", "Fernrohr"),
    ("Teleskop", "Mikroskop"),
]

for a, b in meine_paare:
    print(f"{abstand(a, b):.3f}  {a}  |  {b}")

0.188  Teleskop  |  Fernrohr
0.079  Teleskop  |  Mikroskop


### 4. Die Suche: eine Schleife über alle Chunks

Dieselbe Schleife wie auf der Folie in Java. Sie rechnet für jeden Chunk die Distanz wie in Schritt 3, nur mit dem Vektor des Chunks aus dem Index, sortiert und behält die k nächsten.

In [6]:
def suche(frage, vektoren, thema=None, k=5):
    treffer = []
    for chunk, vektor in zip(chunks, vektoren):
        if thema is not None and chunk["thema"] != thema:
            continue
        kosinus = np.dot(frage, vektor)  # beide Vektoren haben Länge 1: Skalarprodukt = Kosinus
        treffer.append((1 - kosinus, chunk))  # Distanz wie in Java und pgvector: 0 = gleiche Richtung
    treffer.sort(key=lambda t: t[0])
    return treffer[:k]

### 5. Die Suchen aus der Live-Demo

> **Aufgabe 3**: Die vier Suchen aus der Live-Demo lesen, dann in Schritt 6 eine eigene Suche schreiben. Fertig, wenn ihr für eure Suche sagen könnt, ob sie die Frage beantwortet und wie weit der beste Treffer weg liegt, verglichen mit der Dämmerung und dem Universum.

In [7]:
start = time.perf_counter()
frage = einbetten("query: Wann beginnt die astronomische Dämmerung?")
mitte = time.perf_counter()
treffer = suche(frage, vektoren)
ende = time.perf_counter()

zeige(treffer)
print(f"Embedding {(mitte - start) * 1000:.0f} ms, Suche {(ende - mitte) * 1000:.1f} ms über {len(chunks)} Chunks")

1  0.173  Twilight – Astronomical twilight
          Astronomical twilight occurs when the geometric center of the Sun is b …
2  0.186  Twilight – Between day and night
          because after the equinoxes the Sun's declination is then less than 4° …
3  0.196  Twilight – Between day and night
          astronomical dusk in the entire year is 48º33'43". The maximum latitud …
4  0.200  Twilight – Civil twilight
          Civil twilight is the period of time for which the geometric center of …
5  0.203  Twilight – Lasting from one day to the next
          night. Nautical twilight: between about 54°34' and 60°34' north or sou …
Embedding 6 ms, Suche 0.4 ms über 643 Chunks


Der Korpus beantwortet die nächste Frage nicht. Liefert die Suche nichts oder fünf Treffer?

In [8]:
zeige(suche(einbetten("query: Wie alt ist das Universum?"), vektoren))

1  0.206  Planetary nebula – Introduction
          A planetary nebula is a type of emission nebula consisting of an expan …
2  0.207  Globular cluster – Consequences
          The blue stragglers appear on the H–R diagram as a series diverging fr …
3  0.208  Galaxy – Super-luminous spiral
          Recently, researchers described galaxies called super-luminous spirals …
4  0.210  Galaxy – Evolution
          Once stars begin to form, emit radiation, and in some cases explode, t …
5  0.210  Globular cluster – Composition
          Globular clusters are generally composed of hundreds of thousands of l …


Die beiden Suchtexte, die Marvin im ersten Live-Lauf von AstroPlanner geschrieben hat, mit dem Themenfilter `filters` (100 Chunks):

In [9]:
zeige(suche(einbetten("query: OIII Filter"), vektoren, thema="filters"))
print()
zeige(suche(einbetten("query: narrowband OIII filter oxygen emission"), vektoren, thema="filters"))

1  0.159  Astronomical filter – Ultra-high contrast filters
          Known commonly as UHC filters, these filters consist of things which a …
2  0.161  Astronomical filter – Moon filters
          Neutral density filters, also known in astronomy as Moon filters, are …
3  0.167  HOO, SHO and LRGB – Narrowband filters
          Emission nebulae shine in a few spectral lines. The most important for …
4  0.167  HOO, SHO and LRGB – HOO
          HOO uses two filters: Ha is mapped to red, and OIII is mapped to both …
5  0.168  Astronomical filter – Specific emission line filters
          Specific emission line (or lines) filters are used to isolate lines of …

1  0.095  HOO, SHO and LRGB – Narrowband filters
          Emission nebulae shine in a few spectral lines. The most important for …
2  0.100  Astronomical filter – Narrowband
          Narrow-band filters are astronomical filters which transmit only a nar …
3  0.122  Doubly ionized oxygen – Ion
          Its emission of forbidden lin

Bester Treffer im Vergleich mit der Live-Demo:

| Suchtext | Java | hier |
|---|---|---|
| Wann beginnt die astronomische Dämmerung? | Twilight – Astronomical twilight, 0.165 | Twilight – Astronomical twilight, 0.173 |
| Wie alt ist das Universum? | Galaxy – Super-luminous spiral, 0.203 | Planetary nebula – Introduction, 0.206 |
| OIII Filter | Astronomical filter – Ultra-high contrast filters, 0.156 | dasselbe, 0.159 |
| narrowband OIII filter oxygen emission | HOO, SHO and LRGB – Narrowband filters, 0.093 | dasselbe, 0.095 |

Die Distanzen liegen hier um 0,002 bis 0,008 höher, wegen der quantisierten Fassung in Java. Die Reihenfolge bleibt bei drei Suchen gleich. Beim Universum liegen die ersten fünf Treffer so dicht beieinander (0,206 bis 0,210), dass schon diese kleine Abweichung die Reihenfolge ändert; Galaxy – Super-luminous spiral steht hier auf Rang 3.

Die Suche liefert immer k Treffer, auch wenn keiner passt. Ob einer passt, muss die Anwendung entscheiden.

### 6. Eine eigene Suche

Ein eigener Suchtext, Deutsch oder Englisch. Ideen: eine Frage zur Astrofotografie, eine Frage aus eurem Alltag, die der Korpus nicht beantwortet, ein einzelnes Wort, ein Tippfehler. Themen für den Filter: `filters`, `imaging`, `equipment`, `targets`.

Worauf achten: Liegt der beste Treffer näher als bei der Dämmerung (0,173), weiter weg als beim Universum (0,206) oder dazwischen? Und beantwortet er eure Frage?

In [10]:
mein_suchtext = "Welcher Filter hilft gegen Lichtverschmutzung in der Stadt?"
mein_thema = None  # oder z. B. "filters"

zeige(suche(einbetten("query: " + mein_suchtext), vektoren, thema=mein_thema))

1  0.185  Astronomical filter – Broadband
          The broadband, or light pollution reduction (LPR), filters are designe …
2  0.186  Light pollution – Over-illumination
          stray light and energy costs; "Daylight lighting" demanded by citizens …
3  0.193  Light pollution – Effect on astronomy
          by filtering out spectral lines commonly emitted by sodium- and mercur …
4  0.194  Light pollution – Improving lighting fixtures
          capita energy consumption for public lighting in Italy. The same legis …
5  0.194  Light pollution – Effect on astronomy
          Astronomy is very sensitive to light pollution. The night sky viewed f …


## Bonus: ohne Präfixe

e5 erwartet `query: ` vor der Frage und `passage: ` vor dem Chunk. Was passiert, wenn man beide weglässt? Dafür liegt ein zweiter Index im Ordner: dieselben Chunks, ohne `passage: ` eingebettet.

> **Aufgabe 4** (Bonus): Die Zelle unten so ändern, dass die Frage ohne `query: ` eingebettet und im Index `vektoren_ohne` gesucht wird. Vorher vermuten: Ändert sich nur die Distanz oder auch der beste Treffer? Dann mit Schritt 5 vergleichen.

In [11]:
vektoren_ohne = lade_index(praefix=False)

suchtext = "Wann beginnt die astronomische Dämmerung?"
frage = einbetten("query: " + suchtext)  # ändern: ohne Präfix
zeige(suche(frage, vektoren))  # ändern: im Index ohne Präfix

1  0.173  Twilight – Astronomical twilight
          Astronomical twilight occurs when the geometric center of the Sun is b …
2  0.186  Twilight – Between day and night
          because after the equinoxes the Sun's declination is then less than 4° …
3  0.196  Twilight – Between day and night
          astronomical dusk in the entire year is 48º33'43". The maximum latitud …
4  0.200  Twilight – Civil twilight
          Civil twilight is the period of time for which the geometric center of …
5  0.203  Twilight – Lasting from one day to the next
          night. Nautical twilight: between about 54°34' and 60°34' north or sou …


<details><summary>Tipp</summary>

Zwei Stellen, beide mit `# ändern` markiert: `einbetten(suchtext)` ohne `"query: " +`, und in `suche` statt `vektoren` den Index `vektoren_ohne`.

</details>

In [12]:
# @title Lösung (aufklappen)
demo_suchen = [
    ("Wann beginnt die astronomische Dämmerung?", None),
    ("Wie alt ist das Universum?", None),
    ("OIII Filter", "filters"),
    ("narrowband OIII filter oxygen emission", "filters"),
]


def bester(frage, index, thema):
    distanz, chunk = suche(frage, index, thema=thema)[0]
    return f"{distanz:.3f} {chunk['dokument']} – {chunk['abschnitt']}"


for text, thema in demo_suchen:
    print(text)
    print("  beide mit Präfix:   ", bester(einbetten("query: " + text), vektoren, thema))
    print("  beide ohne Präfix:  ", bester(einbetten(text), vektoren_ohne, thema))
    print("  nur Frage ohne:     ", bester(einbetten(text), vektoren, thema))

Wann beginnt die astronomische Dämmerung?
  beide mit Präfix:    0.173 Twilight – Astronomical twilight
  beide ohne Präfix:   0.170 Twilight – Astronomical twilight
  nur Frage ohne:      0.164 Twilight – Astronomical twilight
Wie alt ist das Universum?
  beide mit Präfix:    0.206 Planetary nebula – Introduction
  beide ohne Präfix:   0.197 Globular cluster – Consequences


  nur Frage ohne:      0.206 Globular cluster – Consequences
OIII Filter
  beide mit Präfix:    0.159 Astronomical filter – Ultra-high contrast filters
  beide ohne Präfix:   0.159 Astronomical filter – Ultra-high contrast filters
  nur Frage ohne:      0.167 Astronomical filter – Ultra-high contrast filters
narrowband OIII filter oxygen emission
  beide mit Präfix:    0.095 HOO, SHO and LRGB – Narrowband filters
  beide ohne Präfix:   0.108 HOO, SHO and LRGB – Narrowband filters
  nur Frage ohne:      0.111 HOO, SHO and LRGB – Narrowband filters


Auf diesem Korpus ändern die Präfixe wenig: Bei den drei Suchen mit Antwort bleibt der beste Treffer derselbe Abschnitt, die Distanz verschiebt sich um bis zu 0,016. Beim Universum wechselt der beste Treffer, und ohne beide Präfixe sinkt die Distanz von 0,206 auf 0,197. Die Frage ohne Antwort sieht dann näher aus, und ein Schwellwert, der mit Präfixen eingestellt wurde, passt nicht mehr genau.

Die dritte Zeile ist der häufigere Fehler in der Praxis: Der Index wurde mit `passage: ` gebaut, die Frage vergisst `query: `. Wie gut die Suche dann noch ist, zeigt eine Messung über viele Fragen, nicht ein Beispiel. Das kommt morgen.

## Bonus: ein Schwellwert für die Suche

Die Suche liefert immer fünf Treffer, auch zur Frage nach dem Alter des Universums. Eine Anwendung braucht eine Regel, wann sie sagt: Dazu steht nichts im Korpus. Die einfachste Regel ist ein Schwellwert: Treffer, die weiter weg liegen als `max_distanz`, fallen weg.

> **Aufgabe 5** (Bonus, Knobelaufgabe): `suche_mit_grenze` ergänzen und `MAX_DISTANZ` so wählen, dass die Dämmerung Treffer liefert und das Universum keine. Die Zelle prüft das gleich. Dann die zwei Fragen in der Zelle danach prüfen: Hält euer Wert? Gibt es einen Wert, mit dem alle vier stimmen?

In [13]:
MAX_DISTANZ = 0.30  # ändern


def suche_mit_grenze(frage, vektoren, max_distanz, thema=None, k=5):
    treffer = suche(frage, vektoren, thema=thema, k=k)
    return treffer  # ergänzen: nur die Treffer behalten, die höchstens max_distanz entfernt sind


def pruefe_grenze(funktion, max_distanz, faelle):
    richtig = 0
    for text, antwort_im_korpus in faelle:
        anzahl = len(funktion(einbetten("query: " + text), vektoren, max_distanz))
        ok = (anzahl > 0) == antwort_im_korpus
        richtig += ok
        soll = "mindestens 1" if antwort_im_korpus else "keine"
        print(f"{'✓' if ok else '✗'} {anzahl} Treffer, soll {soll}: {text}")
    print(f"{richtig} von {len(faelle)} wie erwartet")


demo_faelle = [
    ("Wann beginnt die astronomische Dämmerung?", True),
    ("Wie alt ist das Universum?", False),
]
pruefe_grenze(suche_mit_grenze, MAX_DISTANZ, demo_faelle)

✓ 5 Treffer, soll mindestens 1: Wann beginnt die astronomische Dämmerung?
✗ 5 Treffer, soll keine: Wie alt ist das Universum?
1 von 2 wie erwartet


In [14]:
weitere_faelle = [
    ("Was ist Seeing?", True),  # beantwortet der Abschnitt „Astronomical seeing – Introduction“
    ("Wie viel kostet eine Astrokamera?", False),  # Preise stehen nirgends im Korpus
]
pruefe_grenze(suche_mit_grenze, MAX_DISTANZ, weitere_faelle)

✓ 5 Treffer, soll mindestens 1: Was ist Seeing?
✗ 5 Treffer, soll keine: Wie viel kostet eine Astrokamera?
1 von 2 wie erwartet


<details><summary>Tipp 1</summary>

Jeder Treffer ist ein Paar `(distanz, chunk)`, `t[0]` ist die Distanz. Welcher Wert für `MAX_DISTANZ` in Frage kommt, zeigt die Tabelle in Schritt 5: Er muss zwischen dem besten Treffer der Dämmerung und dem des Universums liegen.

</details>
<details><summary>Tipp 2</summary>

Eine Zeile filtert die Liste: `return [t for t in treffer if t[0] <= max_distanz]`. In Java wäre das `treffer.stream().filter(t -> t.distanz() <= maxDistanz).toList()`.

</details>

In [15]:
# @title Lösung (aufklappen)
def suche_mit_grenze_loesung(frage, vektoren, max_distanz, thema=None, k=5):
    treffer = suche(frage, vektoren, thema=thema, k=k)
    return [t for t in treffer if t[0] <= max_distanz]


# Jeder Wert ab 0.173 (Dämmerung) und unter 0.206 (Universum) besteht die erste Prüfung, hier 0.19.
pruefe_grenze(suche_mit_grenze_loesung, 0.19, demo_faelle + weitere_faelle)

print("\nBester Treffer der vier Fragen, nach Distanz sortiert:")
beste = []
for text, antwort_im_korpus in demo_faelle + weitere_faelle:
    distanz, chunk = suche(einbetten("query: " + text), vektoren)[0]
    beste.append((distanz, "Antwort im Korpus" if antwort_im_korpus else "keine Antwort", text))
for distanz, art, text in sorted(beste):
    print(f"{distanz:.3f}  {art:18}  {text}")

✓ 2 Treffer, soll mindestens 1: Wann beginnt die astronomische Dämmerung?
✓ 0 Treffer, soll keine: Wie alt ist das Universum?


✗ 0 Treffer, soll mindestens 1: Was ist Seeing?
✗ 5 Treffer, soll keine: Wie viel kostet eine Astrokamera?
2 von 4 wie erwartet

Bester Treffer der vier Fragen, nach Distanz sortiert:


0.172  keine Antwort       Wie viel kostet eine Astrokamera?
0.173  Antwort im Korpus   Wann beginnt die astronomische Dämmerung?
0.206  keine Antwort       Wie alt ist das Universum?
0.230  Antwort im Korpus   Was ist Seeing?


Kein fester Wert macht alle vier richtig. Die Frage nach dem Preis einer Astrokamera liegt mit 0,172 so nah wie die Dämmerung (0,173): Der Korpus hat einen Abschnitt über Astrofotografie mit Kameras, nur keinen Preis. „Was ist Seeing?“ liegt mit 0,230 weiter weg als das Universum (0,206), obwohl der richtige Abschnitt auf Rang 1 steht.

Die Distanz misst, wie nah eine Frage am Thema eines Chunks liegt, nicht, ob der Chunk sie beantwortet. Ein Schwellwert ist deshalb eine Leitplanke gegen klar fremde Fragen, kein scharfes Kriterium. Ob die Treffer die Frage wirklich beantworten, kann in einer Anwendung das Sprachmodell beurteilen, wenn der Prompt erlaubt, „steht nicht in den Quellen“ zu sagen. Wie weit Fragen mit und ohne Antwort überlappen, zeigt sich erst über viele Fragen. Morgen messt ihr die Suche mit 20 Testfragen und drei Fragen ohne Antwort im Korpus.

## Was ihr gesehen habt

Ein Text wird ein Vektor mit 384 Zahlen, egal wie lang er ist, und zwei Texte sind sich nah, wenn ihre Vektoren in dieselbe Richtung zeigen. Die Suche ist eine Schleife: Distanz zu jedem Chunk, sortieren, die ersten fünf. Sie liefert immer fünf Treffer, auch zur Frage nach dem Alter des Universums, und ob ein Treffer die Frage beantwortet, steht nicht in der Distanz.